In [0]:
%restart_python

In [0]:
import pandas as pd
import numpy as np
import re
import os

import sys
sys.path.append("/Workspace/Users/karthik.raj@bio-techne.com/ScoringPhysics")

In [0]:
dbutils.widgets.text('path_design_campaign', "") # Contains Sergey MPNN AF2 Folders as subfolders
path_design_campaign = dbutils.widgets.get('path_design_campaign')
#path_design_campaign = "/Volumes/sandbox/karthik/motif_scaffolding/experiments/rfdiffusion3/designs/rfd3_test_imesi/"
print(path_design_campaign)

In [0]:
import os
from pathlib import Path
import pandas as pd

paths_mpnn_results = [path for path in Path(path_design_campaign).rglob("mpnn_results.csv")]

list_df_mpnns = []
for path_mpnn_result in paths_mpnn_results:
    path_design_folder = path_mpnn_result.parent
    name_design_folder = path_design_folder.name
    df_design = pd.read_csv(path_mpnn_result, index_col = 0)
    df_design["design_folder_path"] = str(path_design_folder)
    df_design["design_folder_name"] = name_design_folder
    list_df_mpnns.append(df_design)
df_mpnn = pd.concat(list_df_mpnns).reset_index(drop=True)
df_mpnn['design_name'] = df_mpnn['design_folder_name'] + "_design" + df_mpnn['design'].astype(str) + "_n" + df_mpnn['n'].astype(str)
df_mpnn

In [0]:
df_mpnn['contig'].iloc[0]

In [0]:
df_mpnn_seqs = df_mpnn['seq'].str.split('/', expand= True)
df_mpnn_seqs.rename(columns = {0 : 'seq_target', 1: 'seq_binder'}, inplace = True)
df_mpnn_seqs['len_binder'] = df_mpnn_seqs['seq_binder'].str.len()
df_mpnn.insert(loc = 8, column = 'seq_binder', value= df_mpnn_seqs['seq_binder'].values)
df_mpnn.insert(loc = 9, column = 'seq_target', value= df_mpnn_seqs['seq_target'].values)
df_mpnn.head(10)

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
motif_columns = list(df_mpnn.filter(like = "motif").columns)
fig, ax = plt.subplots(len(motif_columns), 1, figsize = (10,10))
for i, column in enumerate(motif_columns):
    sns.histplot(data = df_mpnn, x = column, bins= 50, ax = ax[i], label = column)
    ax[i].set_title(f"{column} distribution")
    ax[i].set_xlabel(column)
    ax[i].set_ylabel("Density")
    ax[i].legend(loc = 'upper right')
    

In [0]:
import seaborn as sns
import matplotlib.pyplot as plt

def plot_histogram_seaborn(df, column, bins=50, figsize=(10, 5), **kwargs):
    fig, ax = plt.subplots(figsize=figsize)
    sns.histplot(df[column], bins=bins, ax=ax, **kwargs)
    ax.set_title(f"{column} distribution")
    ax.set_xlabel(column)
    ax.set_ylabel("Frequency")

column = 'plddt_binder'
plot_histogram_seaborn(df_mpnn, column, bins=50)

In [0]:
agg_funcs = ['mean', 'std', 'count']
if "rfd_model_type" in df_mpnn.columns:
    groupby_cols = ['rfd_model_type', 'contig']
else:
    groupby_cols = ['contig']
confidence_cols = ['rmsd', 'plddt', 'plddt_binder', 'ptm', 'pae']
motif_cols = list(df_mpnn.filter(like = "motif").columns)
agg_dict = {col: agg_funcs for col in confidence_cols + motif_cols}
sort_col = motif_cols[0] if motif_cols else 'rmsd'
print("Sorted by: ", sort_col)
df_mpnn_copy = df_mpnn.copy()
df_mpnn_groupedby = df_mpnn_copy.groupby(groupby_cols).agg(agg_dict).sort_values([(sort_col, 'mean')], ascending= True)
df_mpnn_groupedby.reset_index(inplace = True)
df_mpnn_groupedby

In [0]:
if len(motif_cols) == 2:
    motif_col_y, motif_col_x = motif_cols
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.scatterplot(x = df_mpnn[motif_col_x], y = df_mpnn[motif_col_y], ax = ax, label = f'{motif_col_x} vs {motif_col_y}')

plt.plot([df_mpnn[motif_col_x].min(), df_mpnn[motif_col_x].max()],
         [df_mpnn[motif_col_x].min(), df_mpnn[motif_col_x].max()],
         color='red', linestyle='-', linewidth=2, label='y = x')

plt.title(f"{motif_col_x} vs {motif_col_y}")
plt.xlabel(motif_col_x)
plt.ylabel(motif_col_y)
plt.legend()
plt.show()

### Visualizing RFDiffusion Designed Structures

In [0]:
motif_threshold = {}
threshold_rmsd_holo_binder = 2
for motif_col in motif_cols:
    #df_temp = df_mpnn[df_mpnn[motif_col] >= 0] # Adjust for negative values and ensure the threshold is only based on rmsds that were calculated. Not calculated rmsds set to -1.0
    #threshold = df_temp[motif_col].describe()['50%']
    motif_threshold[motif_col] = threshold_rmsd_holo_binder
    print(f"Threshold for {motif_col}: {threshold_rmsd_holo_binder}")
rmsd_threshold = motif_threshold | {'rmsd_holo_binder_rfd_af2' : threshold_rmsd_holo_binder}
rmsd_threshold

In [0]:
threshold_plddt_binder = 70

filter_rmsd_conditions = ' and '.join([f"{col} < {threshold}" for col, threshold in rmsd_threshold.items()])
filter_plddt_condition = f"plddt_binder > {threshold_plddt_binder} and "
filter_conditions = filter_plddt_condition + filter_rmsd_conditions
print("Filtering based on:", filter_conditions)
df_filtered = df_mpnn.query(filter_conditions).reset_index(drop = True)
print("Number of Passing Designs for Boltz2 Evaluation: ", len(df_filtered))
df_filtered

In [0]:
import py2Dmol
from StrucTools import visualize_structure
import random
df_data = df_filtered.copy()
random_index = random.randint(0, len(df_data) - 1)
#random_index = 100  #(bad design in globular double strand is with index of 4097)
design_path = df_data['design_pdb_path'].iloc[random_index]
print(f"Random index: {random_index}")
print(f"Design Name: {df_data['design_name'].iloc[random_index]}")
print(f"Design Path: {design_path}")
print(f"RMSD: {df_data['rmsd'].iloc[random_index]}")
print(f"RMSD Holo Binder RFD AF2: {df_data['rmsd_holo_binder_rfd_af2'].iloc[random_index]}")
print(f"PLDDT: {df_data['plddt'].iloc[random_index]}")
print(f"PLDDT Binder: {df_data['plddt_binder'].iloc[random_index]}")
print(f"PTM: {df_data['ptm'].iloc[random_index]}")
print(f"PAE: {df_data['pae'].iloc[random_index]}")
print(f"Contigs: {df_data['contig'].iloc[random_index]}")
visualize_structure(design_path)

### Save into Final DataFrame & Split into 4 DataFrames for Parallel Execution

In [0]:
df_filtered_final = df_filtered.reset_index(drop = True)
df_filtered_final.head()

In [0]:
import os
folder_name_af2_outputs = "af2_passed_boltz_input"
path_af2_outputs = os.path.join(path_design_campaign, folder_name_af2_outputs)
if not os.path.exists(path_af2_outputs):
    # 1. Create the folder to store AF2 outputs and analysis 
    os.makedirs(path_af2_outputs)
    print(f"Created folder: {path_af2_outputs}")
else:
    print(f"Folder already exists: {path_af2_outputs}")

# Save the filtered designs to a CSV file
list_paths_af2_outputs = []
path_af2_passed_designs = os.path.join(path_af2_outputs, "af2_passed_designs.csv")
df_filtered_final.to_csv(path_af2_passed_designs)
# If the number of passed designs is greater than 20, split the dataframe into chunks for running Boltz in parallel
if len(df_filtered_final) > 20:
    for index, df_chunk in enumerate(np.array_split(df_filtered_final, 7)):
        path_af2_passed_chunk = os.path.join(path_af2_outputs, f"af2_passed_designs_chunk_{index}.csv")
        df_chunk.to_csv(path_af2_passed_chunk)
        list_paths_af2_outputs.append(path_af2_passed_chunk)
# If the number of passed designs is less than 20, save the dataframe to a single CSV file
else:
    list_paths_af2_outputs.append(path_af2_passed_designs)

# Save list of paths to passed CSV files to dbutils.taskValue
dbutils.jobs.taskValues.set(key = "list_paths_design_csv", value = list_paths_af2_outputs)

# Decide whether to run Boltz2 Holo or Not
if len(df_filtered_final) == 0:
    dbutils.jobs.taskValues.set(key = "run_boltz", value = "")
    print("No designs passed the filter. Skipping Boltz2 Holo.")
else:
    dbutils.jobs.taskValues.set(key = "run_boltz", value = "holo")
    print("Designs passed the filter. Running Boltz2 Holo.")